In [126]:
import numpy as np
import pandas as pd

In [127]:
path = "../dataset/AzureFunctionsInvocationTraceForTwoWeeksJan2021.txt"
df = pd.read_csv(path)

In [128]:
df['arrival_time'] = df['end_timestamp'] - df['duration']
df.sort_values(by=['arrival_time'], inplace=True, ascending=True)
df['arrival_time_fix'] = df['arrival_time'] - df['arrival_time'].iloc[0]
df.sort_values(by=['arrival_time_fix'], inplace=True, ascending=True)

In [129]:
#first we add divide the dataset into 4 hour windows
hour_trace = 4
df['trace_id'] = (df['arrival_time'] // (hour_trace * 60 * 60)).astype(int)
df

,app,func,end_timestamp,duration,arrival_time,arrival_time_fix,trace_id
0,7b2c43a2bc30f6bb438074df88b603d2cb982d3e7961de...,e3cdb48830f66eb8689cc0223514569a69812b77e6611e...,7.949090e-02,0.078,1.490900e-03,0.000000e+00,0
1,1573b95c039e51cc012b543a4af3bc7c3ee9485acbb003...,337cd24a7d5fd5c92460faee4ebe6a186a0eb322bd17b7...,5.715786e+01,57.154,3.860041e-03,2.369141e-03,0
2,1573b95c039e51cc012b543a4af3bc7c3ee9485acbb003...,48cc770d590d3c5a7691b3b4e9302f82ec3be5ddc2a037...,5.913048e+01,59.125,5.477905e-03,3.987005e-03,0
3,f274d71de386ccc77e4ca74766dbc485461c3053059d47...,3d2aee54a133509f16fb636d74128c2adcfcac71c6dcef...,6.252541e+00,6.236,1.654107e-02,1.505017e-02,0
4,7b2c43a2bc30f6bb438074df88b603d2cb982d3e7961de...,68bbfd828223a505d7917339f4656c5f33ff93225cdb9d...,6.682396e-02,0.050,1.682396e-02,1.533306e-02,0
...,...,...,...,...,...,...,...
1980946,a594f92f84072b4cd031fe5283d1781a6e98f430696dec...,155e47f8e7f751d0c845049456d01832013c61336a8cd8...,1.209597e+06,0.001,1.209597e+06,1.209597e+06,83
1980947,a594f92f84072b4cd031fe5283d1781a6e98f430696dec...,155e47f8e7f751d0c845049456d01832013c61336a8cd8...,1.209598e+06,0.001,1.209598e+06,1.209598e+06,83
1980948,a594f92f84072b4cd031fe5283d1781a6e98f430696dec...,155e47f8e7f751d0c845049456d01832013c61336a8cd8...,1.209599e+06,0.001,1.209599e+06,1.209599e+06,83
1980949,a594f92f84072b4cd031fe5283d1781a6e98f430696dec...,155e47f8e7f751d0c845049456d01832013c61336a8cd8...,1.209599e+06,0.001,1.209599e+06,1.209599e+06,83


In [130]:

df['minute_bin'] = (df['arrival_time'] // 60).astype(int)

In [131]:
# we will only be calculating for
max_trace_value = df['trace_id'].max()
all_cov_dfs = []

for l in range(max_trace_value+1):
    sf = df[df['trace_id'] == l]
    requests_per_minute = sf.groupby(['func', 'minute_bin']).size().reset_index(name='request_count')
    pivot_df = requests_per_minute.pivot_table(index='minute_bin', columns='func', values='request_count', fill_value=0)
    cov_per_function = pivot_df.std(ddof=0) / pivot_df.mean()
    total_requests_per_function = sf.groupby('func').size()
    cov_df = pd.DataFrame({
            'func': cov_per_function.index,
            'CoV': cov_per_function.values,
            'total_requests': total_requests_per_function[cov_per_function.index].values,
            'trace_id': l
        })
    cov_df = cov_df[cov_df['total_requests'] > 50]
    all_cov_dfs.append(cov_df)

    
    

In [132]:
final_cov_df = pd.concat(all_cov_dfs, ignore_index=True)
final_cov_df

,func,CoV,total_requests,trace_id
0,155e47f8e7f751d0c845049456d01832013c61336a8cd8...,1.334483,8967,0
1,313c03f53a0d31f70aec25f62efb33e7dd779725ca4af5...,0.540062,240,0
2,31aa5ab69d7730086b08f4303aa2ac3244c639866908fc...,1.823574,3206,0
3,34f4775366e51728635af48df1a96d332cf1565eee069a...,1.414214,80,0
4,426930ee2324a425c25463ff35403f47f2f86624df705f...,2.232390,644,0
...,...,...,...,...
1594,cc5bb2108cc7daf53f9728ad21f661a8ef9c8b36284bac...,1.118631,4240,83
1595,e52b340be2bf2050fd4811834485a98e6ab6dc8374d138...,1.000000,120,83
1596,f59e480aa82338034326aebcef6c9c43cb63e5848f98c2...,1.000000,120,83
1597,f5b7a048365a1ee797a55074a5b7145d285623463d3fcd...,1.000000,120,83


In [133]:
# Step 1: Filter where CoV > 4 and total_requests < 1000
filtered_df = final_cov_df[(final_cov_df['CoV'] > 1) & (final_cov_df['CoV'] < 4) & (final_cov_df['total_requests'] < 1000)]

# Step 2: Sort by total_requests descending
filtered_df = filtered_df.sort_values(by='total_requests', ascending=True)

# Step 3: Get only one row per function (the one with highest total_requests under 1000)
top_windows_per_func = filtered_df.drop_duplicates(subset='func', keep='first')

top_windows_per_func

,func,CoV,total_requests,trace_id
718,b3bad0515d801a1abd0cf4a8a11fa96105bb4a7cf42768...,2.464824,51,39
721,e02465de583b6ceffa5b78cce5f10eb27e714a8a6b3aed...,1.919967,51,39
413,d880a3e25b4baf00977b29491a01e9cf3bfd963a31b201...,1.925067,51,21
1173,6b2db95773685aa6d1f646584bbe1252579a0513df8057...,2.819646,51,67
952,cd9f7f333d59aede8088772edac3bff78c45e2dad8eb49...,1.925067,51,54
...,...,...,...,...
760,426930ee2324a425c25463ff35403f47f2f86624df705f...,2.740503,478,42
25,b74c4ab0e0a6349700abbf5ca5f97d54005710f2289f96...,3.462772,526,1
774,31aa5ab69d7730086b08f4303aa2ac3244c639866908fc...,3.053000,612,43
1433,6d827795dde8df95689d3c2319fcd1eb343dd757585a34...,3.170796,893,78


In [134]:
#Since there are too much request
# We will manually select some function.
first_two_row = top_windows_per_func[(top_windows_per_func['total_requests'] > 70) & (top_windows_per_func['total_requests'] < 100)].head(2)
second_two_row = top_windows_per_func[(top_windows_per_func['total_requests'] > 100) & (top_windows_per_func['total_requests'] < 150)].head(2)
third_three_row = top_windows_per_func[(top_windows_per_func['total_requests'] > 150) & (top_windows_per_func['total_requests'] < 700)].head(3)
normal_functions_list = [first_two_row, second_two_row, third_three_row]
normal_functions = pd.concat(normal_functions_list, axis=0, ignore_index=True)
normal_functions

,func,CoV,total_requests,trace_id
0,1cf618f9ee318fecbc31aa059ded8328581b9a9b092f10...,2.563187,71,14
1,cc5bb2108cc7daf53f9728ad21f661a8ef9c8b36284bac...,3.530714,73,5
2,ce872156aeace3e8141b278f60c55f3eb344bf25cc612e...,3.941139,106,23
3,1bfbf6e1f849d22a0ece669507eb055525edf6929f273b...,2.705942,107,14
4,bd34d747bf67247a686b958802ebe65c048d6d3af170f2...,2.603168,155,77
5,e40593c3e6576988521de38142553e9f894d32374d11a0...,1.979150,157,82
6,ec5e0d5b0edaab92fc4bc2ae62461a9068673e155929f1...,3.872704,172,24


In [135]:

normal_functions['id'] = normal_functions['func']
normal_functions['func'] = ['alexnet','efficientnet','bert','distilgpt2','resnet50','googlenet','inception']
normal_functions['memory'] = 1024
normal_functions['filepath'] = ['alexnet.py','efficientnet.py','bert.py','distilgpt2.py','resnet50.py','googlenet.py','inception.py']
normal_functions.to_csv('../dataset/normal_functions.csv', index=False)

In [136]:
# Step 1: Filter where CoV > 4 and total_requests < 1000
filtered_df1 = final_cov_df[(final_cov_df['CoV'] > 4)  & (final_cov_df['total_requests'] < 1000)]

# Step 2: Sort by total_requests descending
filtered_df1 = filtered_df1.sort_values(by='total_requests', ascending=True)

# filtered_df
# Step 3: Get only one row per function (the one with highest total_requests under 1000)
top_windows_per_func1 = filtered_df1.drop_duplicates(subset='func', keep='first')
# brusty_functions = top_windows_per_func.head(7)
# brusty_functions['id'] = brusty_functions['func']
# brusty_functions['func'] = ['resnet50','googlenet','inception','alexnet','efficientnet','bert','distilgpt2']
# brusty_functions['memory'] = 1024
# brusty_functions['filepath'] = ['resnet50.py','googlenet.py','inception.py','alexnet.py','efficientnet.py','bert.py','distilgpt2.py']

# brusty_functions.to_csv('../dataset/brusty_functions.csv', index=False)


In [137]:
#Since there are too much request
# We will manually select some function.
# first_two_row = top_windows_per_func[(top_windows_per_func['total_requests'] > 60) & (top_windows_per_func['total_requests'] < 100)].head(2)
# second_two_row = top_windows_per_func[(top_windows_per_func['total_requests'] > 100) & (top_windows_per_func['total_requests'] < 250)].head(2)
# third_three_row = top_windows_per_func[(top_windows_per_func['total_requests'] > 470) & (top_windows_per_func['total_requests'] < 700)].head(3)


first_two_row1 = top_windows_per_func1[(top_windows_per_func1['total_requests'] > 60) & (top_windows_per_func1['total_requests'] < 100)].head(2)
second_two_row1 = top_windows_per_func1[(top_windows_per_func1['total_requests'] > 110) & (top_windows_per_func1['total_requests'] < 150)].head(2)
third_three_row1 = top_windows_per_func1[(top_windows_per_func1['total_requests'] > 150) & (top_windows_per_func1['total_requests'] < 900)].head(3)
brusty_functions_list1 = [first_two_row1, second_two_row1, third_three_row1]
brusty_functions = pd.concat(brusty_functions_list1, axis=0, ignore_index=True)
brusty_functions

,func,CoV,total_requests,trace_id
0,ec5e0d5b0edaab92fc4bc2ae62461a9068673e155929f1...,5.784816,62,40
1,e09dddbcb72e0cbc5c9b775bdfd4dce5913b972c3a71eb...,9.707631,63,53
2,155e47f8e7f751d0c845049456d01832013c61336a8cd8...,11.110848,111,12
3,50242e50bdc20d4a9468e40a01658f98198029fef10071...,15.459625,112,65
4,eafaad5acf5d68a876c0fe609024baa3e901700c7a19a8...,5.071702,155,15
5,ae6f97c69d684b23e45ce195db1da6d3853004596e39b7...,10.264735,218,52
6,30aa434528bc68ee07745ee7be3a0bdb33d58961fdc846...,7.755490,263,55


In [138]:
brusty_functions['id'] = brusty_functions['func']
brusty_functions['func'] = ['alexnet','efficientnet','bert','distilgpt2','resnet50','googlenet','inception']
brusty_functions['memory'] = 1024
brusty_functions['filepath'] = ['alexnet.py','efficientnet.py','bert.py','distilgpt2.py','resnet50.py','googlenet.py','inception.py']
brusty_functions.to_csv('../dataset/brusty_functions.csv', index=False)

In [139]:
# Step 1: Create empty list
all_brusty_df = [] 

# Step 2: Loop and collect tf DataFrames
for index, row in brusty_functions.iterrows():
    tf = df[(df['func'] == row['id']) & (df['trace_id'] == row['trace_id'])]
    all_brusty_df.append(tf)

# Step 3: Concatenate all tf into one DataFrame
merged_brusty_tf = pd.concat(all_brusty_df, ignore_index=True)
merged_brusty_tf['arrival_time'] = merged_brusty_tf['arrival_time'] - (merged_brusty_tf['trace_id'] * 60 * 60 * 4)
# merged_brusty_tf['arrival_time'] = merged_brusty_tf['arrival_time'].apply(lambda x: f"{x:.0f}")
merged_brusty_tf = merged_brusty_tf.sort_values(by='arrival_time', ascending=True)

# Here we need to reset the 
merged_brusty_tf.to_csv('../dataset/brusty_df.csv', index=False)

In [140]:
# Step 1: Create empty list
all_normal_df = [] 

# Step 2: Loop and collect tf DataFrames
for index, row in normal_functions.iterrows():
    tf1 = df[(df['func'] == row['id']) & (df['trace_id'] == row['trace_id'])]
    all_normal_df.append(tf1)

# Step 3: Concatenate all tf into one DataFrame
merged_normal_tf = pd.concat(all_normal_df, ignore_index=True)
merged_normal_tf['arrival_time'] = merged_normal_tf['arrival_time'] - (merged_normal_tf['trace_id'] * 60 * 60 * 4)
# merged_normal_tf['arrival_time'] = merged_normal_tf['arrival_time'].apply(lambda x: f"{x:.0f}")
merged_normal_tf = merged_normal_tf.sort_values(by='arrival_time', ascending=True)

# Here we need to reset the 
merged_normal_tf.to_csv('../dataset/normal_df.csv', index=False)

In [141]:
merged_normal_tf

,app,func,end_timestamp,duration,arrival_time,arrival_time_fix,trace_id,minute_bin
144,37a025445321ed431e18bd48a4770a00e4adce4bd3a7a3...,ce872156aeace3e8141b278f60c55f3eb344bf25cc612e...,331221.612795,6.087,15.525795,331215.524304,23,5520
145,37a025445321ed431e18bd48a4770a00e4adce4bd3a7a3...,ce872156aeace3e8141b278f60c55f3eb344bf25cc612e...,331221.474273,5.706,15.768273,331215.766782,23,5520
146,37a025445321ed431e18bd48a4770a00e4adce4bd3a7a3...,ce872156aeace3e8141b278f60c55f3eb344bf25cc612e...,331235.923544,14.242,21.681544,331221.680053,23,5520
147,37a025445321ed431e18bd48a4770a00e4adce4bd3a7a3...,ce872156aeace3e8141b278f60c55f3eb344bf25cc612e...,331241.364920,5.372,35.992920,331235.991429,23,5520
148,37a025445321ed431e18bd48a4770a00e4adce4bd3a7a3...,ce872156aeace3e8141b278f60c55f3eb344bf25cc612e...,331243.973818,7.860,36.113818,331236.112327,23,5520
...,...,...,...,...,...,...,...,...
68,96149d3ed4f00afb92f12856101e693e93c4683f030ae6...,1cf618f9ee318fecbc31aa059ded8328581b9a9b092f10...,215759.974498,0.046,14159.928498,215759.927007,14,3595
355,96149d3ed4f00afb92f12856101e693e93c4683f030ae6...,1bfbf6e1f849d22a0ece669507eb055525edf6929f273b...,215762.092330,1.298,14160.794330,215760.792839,14,3596
69,96149d3ed4f00afb92f12856101e693e93c4683f030ae6...,1cf618f9ee318fecbc31aa059ded8328581b9a9b092f10...,215762.639565,0.043,14162.596565,215762.595074,14,3596
356,96149d3ed4f00afb92f12856101e693e93c4683f030ae6...,1bfbf6e1f849d22a0ece669507eb055525edf6929f273b...,215960.753888,0.990,14359.763888,215959.762397,14,3599


In [142]:
merged_brusty_tf

,app,func,end_timestamp,duration,arrival_time,arrival_time_fix,trace_id,minute_bin
503,d399b29c39aafde69884d67ced0990a879c305854eca7e...,ae6f97c69d684b23e45ce195db1da6d3853004596e39b7...,749247.263969,4.156,443.107969,749243.106478,52,12487
504,d399b29c39aafde69884d67ced0990a879c305854eca7e...,ae6f97c69d684b23e45ce195db1da6d3853004596e39b7...,749483.821926,0.671,683.150926,749483.149435,52,12491
505,d399b29c39aafde69884d67ced0990a879c305854eca7e...,ae6f97c69d684b23e45ce195db1da6d3853004596e39b7...,749541.777182,0.562,741.215182,749541.213691,52,12492
506,d399b29c39aafde69884d67ced0990a879c305854eca7e...,ae6f97c69d684b23e45ce195db1da6d3853004596e39b7...,749722.025895,0.942,921.083895,749721.082404,52,12495
236,a0ed6d33dc4622e0a557f45a85a64d359ca20c3e5888fe...,50242e50bdc20d4a9468e40a01658f98198029fef10071...,937025.313915,4.195,1021.118915,937021.117424,65,15617
...,...,...,...,...,...,...,...,...
982,70b9cea7ca266637479483f517194c402dfe99b5fc2357...,30aa434528bc68ee07745ee7be3a0bdb33d58961fdc846...,801382.752142,29.685,9353.067142,801353.065651,55,13355
983,70b9cea7ca266637479483f517194c402dfe99b5fc2357...,30aa434528bc68ee07745ee7be3a0bdb33d58961fdc846...,801368.507839,15.415,9353.092839,801353.091348,55,13355
500,62ed48c098820db02aa8e99ad41e5438e61334ba7b1618...,eafaad5acf5d68a876c0fe609024baa3e901700c7a19a8...,226825.968600,6.896,10819.072600,226819.071109,15,3780
501,62ed48c098820db02aa8e99ad41e5438e61334ba7b1618...,eafaad5acf5d68a876c0fe609024baa3e901700c7a19a8...,226826.086515,6.737,10819.349515,226819.348024,15,3780


In [122]:
brusty_functions

,func,CoV,total_requests,trace_id,id,memory,filepath
0,alexnet,5.784816,62,40,ec5e0d5b0edaab92fc4bc2ae62461a9068673e155929f1...,1024,alexnet.py
1,efficientnet,9.707631,63,53,e09dddbcb72e0cbc5c9b775bdfd4dce5913b972c3a71eb...,1024,efficientnet.py
2,bert,11.110848,111,12,155e47f8e7f751d0c845049456d01832013c61336a8cd8...,1024,bert.py
3,distilgpt2,15.459625,112,65,50242e50bdc20d4a9468e40a01658f98198029fef10071...,1024,distilgpt2.py
4,resnet50,5.071702,155,15,eafaad5acf5d68a876c0fe609024baa3e901700c7a19a8...,1024,resnet50.py
5,googlenet,10.264735,218,52,ae6f97c69d684b23e45ce195db1da6d3853004596e39b7...,1024,googlenet.py
6,inception,7.755490,263,55,30aa434528bc68ee07745ee7be3a0bdb33d58961fdc846...,1024,inception.py


In [123]:
normal_functions

,func,CoV,total_requests,trace_id,id,memory,filepath
0,alexnet,1.801549,62,75,e52b340be2bf2050fd4811834485a98e6ab6dc8374d138...,1024,alexnet.py
1,efficientnet,3.953337,63,58,4d0668e4dc51e885b3cbd1d1bc73be7bc81734cb91ea8b...,1024,efficientnet.py
2,bert,3.301555,116,39,2845f72936c1c598cda9b18718b8621c839796c90adb3e...,1024,bert.py
3,distilgpt2,1.031485,122,8,c9f8e30e36d1aef62c10b3cfca6e289a93848a148d876d...,1024,distilgpt2.py
4,resnet50,2.603168,155,77,bd34d747bf67247a686b958802ebe65c048d6d3af170f2...,1024,resnet50.py
5,googlenet,1.979150,157,82,e40593c3e6576988521de38142553e9f894d32374d11a0...,1024,googlenet.py
6,inception,3.872704,172,24,ec5e0d5b0edaab92fc4bc2ae62461a9068673e155929f1...,1024,inception.py


In [124]:
merged_brusty_tf

,app,func,end_timestamp,duration,arrival_time,arrival_time_fix,trace_id,minute_bin
503,d399b29c39aafde69884d67ced0990a879c305854eca7e...,ae6f97c69d684b23e45ce195db1da6d3853004596e39b7...,749247.263969,4.156,443.107969,749243.106478,52,12487
504,d399b29c39aafde69884d67ced0990a879c305854eca7e...,ae6f97c69d684b23e45ce195db1da6d3853004596e39b7...,749483.821926,0.671,683.150926,749483.149435,52,12491
505,d399b29c39aafde69884d67ced0990a879c305854eca7e...,ae6f97c69d684b23e45ce195db1da6d3853004596e39b7...,749541.777182,0.562,741.215182,749541.213691,52,12492
506,d399b29c39aafde69884d67ced0990a879c305854eca7e...,ae6f97c69d684b23e45ce195db1da6d3853004596e39b7...,749722.025895,0.942,921.083895,749721.082404,52,12495
236,a0ed6d33dc4622e0a557f45a85a64d359ca20c3e5888fe...,50242e50bdc20d4a9468e40a01658f98198029fef10071...,937025.313915,4.195,1021.118915,937021.117424,65,15617
...,...,...,...,...,...,...,...,...
982,70b9cea7ca266637479483f517194c402dfe99b5fc2357...,30aa434528bc68ee07745ee7be3a0bdb33d58961fdc846...,801382.752142,29.685,9353.067142,801353.065651,55,13355
983,70b9cea7ca266637479483f517194c402dfe99b5fc2357...,30aa434528bc68ee07745ee7be3a0bdb33d58961fdc846...,801368.507839,15.415,9353.092839,801353.091348,55,13355
500,62ed48c098820db02aa8e99ad41e5438e61334ba7b1618...,eafaad5acf5d68a876c0fe609024baa3e901700c7a19a8...,226825.968600,6.896,10819.072600,226819.071109,15,3780
501,62ed48c098820db02aa8e99ad41e5438e61334ba7b1618...,eafaad5acf5d68a876c0fe609024baa3e901700c7a19a8...,226826.086515,6.737,10819.349515,226819.348024,15,3780
